<a href="https://colab.research.google.com/github/yejinPARK48/michigan_building_detection/blob/main/01_data_preparation/metadata_add_geo_columns.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Metadata Update — Add Geographic Columns

Adds `state_fips`, `state_name`, `county_fips`, `county_name` to `tile_metadata.csv`.
Based on existing `geoid` (12-digit Census block group GEOID).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import pandas as pd
import os

BASE_DIR = '/content/drive/MyDrive/michigan_unet_project'
META_DIR = f'{BASE_DIR}/metadata'
META_PATH = f'{META_DIR}/tile_metadata.csv'

df_meta = pd.read_csv(META_PATH, dtype={'geoid': str, 'tract_id': str})
print(f'Loaded: {len(df_meta)} rows')
print(df_meta.columns.tolist())


Loaded: 8308 rows
['tile_id', 'geoid', 'county_fips', 'county_type', 'tract_ce', 'blkgrp_ce', 'tract_id', 'namelsad', 'bg_area_km2', 'bbox_minx', 'bbox_miny', 'bbox_maxx', 'bbox_maxy', 'footprint_count', 'fp_area_mean_m2', 'fp_area_sum_m2', 'fp_area_max_m2', 'mask_building_ratio', 'mask_building_px', 'clean', 'reason']


In [ ]:
puma_df = pd.read_excel(f'/content/drive/Shareddrives/ISR-NCSES Project/PUMA Selection/mi_puma_county_indicator/mi_puma_county_indicator.xlsx')
print(puma_df[puma_df['multi_county_PUMA_indicator'] == 1]['county_name'].unique())

['Baraga' 'Dickinson' 'Gogebic' 'Houghton' 'Iron' 'Keweenaw' 'Marquette'
 'Ontonagon' 'Alger' 'Chippewa' 'Delta' 'Luce' 'Mackinac' 'Menominee'
 'Schoolcraft' 'Alcona' 'Alpena' 'Cheboygan' 'Crawford' 'Montmorency'
 'Oscoda' 'Otsego' 'Presque Isle' 'Antrim' 'Charlevoix' 'Emmet' 'Kalkaska'
 'Missaukee' 'Wexford' 'Benzie' 'Grand Traverse' 'Leelanau' 'Manistee'
 'Lake' 'Mason' 'Newaygo' 'Oceana' 'Ionia' 'Mecosta' 'Montcalm' 'Osceola'
 'Clare' 'Gratiot' 'Isabella' 'Arenac' 'Gladwin' 'Iosco' 'Ogemaw'
 'Roscommon' 'Bay' 'Midland' 'Huron' 'Sanilac' 'Tuscola' 'Genesee'
 'Lapeer' 'Shiawassee' 'Clinton' 'Eaton' 'Barry' 'Calhoun' 'Branch'
 'St. Joseph' 'Cass' 'Van Buren' 'Hillsdale' 'Lenawee']


In [ ]:
# ── Add geographic columns ───────────────────────────────────────────────
county_names = {
    '26163': 'Wayne',      '26125': 'Oakland',    '26049': 'Genesee',
    '26139': 'Ottawa',     '26155': 'Shiawassee', '26073': 'Isabella',
    '26117': 'Montcalm',   '26103': 'Marquette',  '26085': 'Lake',
    '26131': 'Ontonagon',  '26053': 'Gogebic',    '26061': 'Houghton',
    '26071': 'Iron',       '26013': 'Baraga',     '26157': 'Tuscola',
    '26151': 'Sanilac',    '26063': 'Huron',      '26051': 'Gladwin',
    '26059': 'Hillsdale',
}

df_meta['state_fips']  = '26'
df_meta['state_name']  = 'Michigan'
df_meta['county_fips'] = df_meta['geoid'].str[:5]
df_meta['county_name'] = df_meta['county_fips'].map(county_names)
df_meta['tract_fips']  = df_meta['geoid'].str[:11]
df_meta['bg_id']       = df_meta['geoid'].str[11:]

print('Added columns: state_fips, state_name, county_fips, county_name, tract_fips, bg_id')
print(f'Unmapped counties: {df_meta[df_meta.county_name.isna()].county_fips.unique()}')
print(df_meta[['tile_id', 'geoid', 'state_fips', 'state_name', 'county_fips', 'county_name', 'tract_fips', 'bg_id']].head(10).to_string(index=False))

Added columns: state_fips, state_name, county_fips, county_name, tract_fips, bg_id
Unmapped counties: ['26057' '26043' '26083' '26035' '26087']
           tile_id        geoid state_fips state_name county_fips county_name  tract_fips bg_id
 260639509004_1332 260639509004         26   Michigan       26063       Huron 26063950900     4
260639900000_13492 260639900000         26   Michigan       26063       Huron 26063990000     0
  260639506002_591 260639506002         26   Michigan       26063       Huron 26063950600     2
 260639509003_1315 260639509003         26   Michigan       26063       Huron 26063950900     3
  260639509003_606 260639509003         26   Michigan       26063       Huron 26063950900     3
260639900000_15744 260639900000         26   Michigan       26063       Huron 26063990000     0
 260639506002_1132 260639506002         26   Michigan       26063       Huron 26063950600     2
260639900000_16449 260639900000         26   Michigan       26063       Huron 2606399000

In [ ]:
STATS_PATH = f'{BASE_DIR}/curated/tile_stats_clean.csv'

df_stats = pd.read_csv(STATS_PATH, dtype={'tile_id': str})

county_names = {
    '26163': 'Wayne',      '26125': 'Oakland',    '26049': 'Genesee',
    '26139': 'Ottawa',     '26155': 'Shiawassee', '26073': 'Isabella',
    '26117': 'Montcalm',   '26103': 'Marquette',  '26085': 'Lake',
    '26131': 'Ontonagon',  '26053': 'Gogebic',    '26061': 'Houghton',
    '26071': 'Iron',       '26013': 'Baraga',     '26157': 'Tuscola',
    '26151': 'Sanilac',    '26063': 'Huron',      '26051': 'Gladwin',
    '26059': 'Hillsdale',
}

df_meta_geo = pd.read_csv(f'{BASE_DIR}/metadata/tile_metadata.csv',
                           dtype={'geoid': str})[['tile_id', 'geoid']]
df_stats = df_stats.merge(df_meta_geo, on='tile_id', how='left')

df_stats['state_fips']  = '26'
df_stats['state_name']  = 'Michigan'
df_stats['county_fips'] = df_stats['geoid'].str[:5]
df_stats['county_name'] = df_stats['county_fips'].map(county_names)
df_stats['tract_fips']  = df_stats['geoid'].str[:11]
df_stats['bg_id']       = df_stats['geoid'].str[11:]

df_stats.to_csv(STATS_PATH, index=False)
print(f'Saved: {STATS_PATH}')
print(df_stats[['tile_id', 'geoid', 'county_fips', 'county_name', 'tract_fips', 'bg_id']].head(5).to_string(index=False))

Saved: /content/drive/MyDrive/michigan_unet_project/curated/tile_stats_clean.csv
           tile_id        geoid county_fips county_name  tract_fips bg_id
 260130001001_1004 260130001001       26013      Baraga 26013000100     1
260130001001_10087 260130001001       26013      Baraga 26013000100     1
 260130001001_1069 260130001001       26013      Baraga 26013000100     1
 260130001001_1150 260130001001       26013      Baraga 26013000100     1
260130001001_12422 260130001001       26013      Baraga 26013000100     1


In [ ]:
# ── Save ─────────────────────────────────────────────────────────────────
df_meta.to_csv(META_PATH, index=False)
print(f'Saved: {META_PATH}')
print(f'Total rows: {len(df_meta)}')
print(f'Columns: {df_meta.columns.tolist()}')


Saved: /content/drive/MyDrive/michigan_unet_project/metadata/tile_metadata.csv
Total rows: 8308
Columns: ['tile_id', 'geoid', 'county_fips', 'county_type', 'tract_ce', 'blkgrp_ce', 'tract_id', 'namelsad', 'bg_area_km2', 'bbox_minx', 'bbox_miny', 'bbox_maxx', 'bbox_maxy', 'footprint_count', 'fp_area_mean_m2', 'fp_area_sum_m2', 'fp_area_max_m2', 'mask_building_ratio', 'mask_building_px', 'clean', 'reason', 'state_fips', 'state_name', 'county_name', 'tract_fips', 'bg_id']


In [ ]:
CURATED_DIR = f'{BASE_DIR}/curated'
STATS_PATH  = f'{CURATED_DIR}/tile_stats_clean.csv'

# Geographic columns
county_names = {
    '26163': 'Wayne',      '26125': 'Oakland',    '26049': 'Genesee',
    '26139': 'Ottawa',     '26155': 'Shiawassee', '26073': 'Isabella',
    '26117': 'Montcalm',   '26103': 'Marquette',  '26085': 'Lake',
    '26131': 'Ontonagon',  '26053': 'Gogebic',    '26061': 'Houghton',
    '26071': 'Iron',       '26013': 'Baraga',     '26157': 'Tuscola',
    '26151': 'Sanilac',    '26063': 'Huron',      '26051': 'Gladwin',
    '26059': 'Hillsdale',
}

df_meta['state_fips']  = '26'
df_meta['state_name']  = 'Michigan'
df_meta['county_fips'] = df_meta['geoid'].str[:5]
df_meta['county_name'] = df_meta['county_fips'].map(county_names)
df_meta['tract_fips']  = df_meta['geoid'].str[:11]
df_meta['bg_id']       = df_meta['geoid'].str[11:]

# Tier
df_stats = pd.read_csv(STATS_PATH, dtype={'tile_id': str})
df_meta  = df_meta.merge(df_stats[['tile_id', 'tier']], on='tile_id', how='left')

# Save
df_meta.to_csv(META_PATH, index=False)
print(f'Saved! Columns: {df_meta.columns.tolist()}')
print(df_meta['tier'].value_counts().to_string())

Saved! Columns: ['tile_id', 'geoid', 'county_fips', 'county_type', 'tract_ce', 'blkgrp_ce', 'tract_id', 'namelsad', 'bg_area_km2', 'bbox_minx', 'bbox_miny', 'bbox_maxx', 'bbox_maxy', 'footprint_count', 'fp_area_mean_m2', 'fp_area_sum_m2', 'fp_area_max_m2', 'mask_building_ratio', 'mask_building_px', 'clean', 'reason', 'state_fips', 'state_name', 'county_name', 'tract_fips', 'bg_id', 'tier']
tier
empty     4010
sparse    2490
dense      962


In [ ]:
puma_df = pd.read_excel(f'/content/drive/Shareddrives/ISR-NCSES Project/PUMA Selection/mi_puma_county_indicator/mi_puma_county_indicator.xlsx')

# 19 pumas
our_counties = list(county_names.keys())  # 26XXX
puma_df['county_fips_full'] = '26' + puma_df['county_fips'].astype(str).str.zfill(3)
our_puma = puma_df[puma_df['county_fips_full'].isin(our_counties)]

# number of pumas for each county
county_puma_count = our_puma.groupby('county_fips_full')['puma_code'].nunique()
print(county_puma_count[county_puma_count > 1])

Series([], Name: puma_code, dtype: int64)
